In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import gc

# Verificar GPU
if torch.cuda.is_available():
    print(f"GPU Detectada: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB\n")
else:
    print("GPU NÃO detectada! Rodará em CPU (muito lento)")

# LIMPAR MEMÓRIA
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# 1. MODELO
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# DATASET
print("\nCarregando dataset...")
dataset = load_dataset("imdb", split="train[:10000]")

def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)

print(f"Dataset: {len(tokenized_dataset['train'])} treino / {len(tokenized_dataset['test'])} teste")

#TREINAMENTO (10 EPOCAS)
training_args = TrainingArguments(
    output_dir="./lora-output-v4.2",
    num_train_epochs=10,
    per_device_train_batch_size=8 if torch.cuda.is_available() else 2,
    gradient_accumulation_steps=2 if torch.cuda.is_available() else 8,

    learning_rate=5e-6,
    max_grad_norm=0.3,

    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=3,

    fp16=torch.cuda.is_available(),
    report_to=["none"],
    remove_unused_columns=False,
)

# TREINO
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)

print("\n" + "="*60)
print(f"Dataset: 10,000 exemplos")
print(f"Epocas: 10")

trainer.train()

#SALVAR
model.save_pretrained("./lora-model")
tokenizer.save_pretrained("./lora-model")

print("\n" + "="*60)
print(f" Modelo salvo em: './lora-model'")

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Carregando modelo")

# Verificar se já existe modelo carregado na memória
if 'inference_model' in globals():
    print("Modelo já carregado! Pulando...")
else:
    # Carregar modelo base
    base_model = AutoModelForCausalLM.from_pretrained(
        "gpt2",
        device_map="auto",
        torch_dtype=torch.float16
    )

    # Carregar adapters LoRA
    inference_model = PeftModel.from_pretrained(base_model, "./lora_outputs/lora-model")
    inference_tokenizer = AutoTokenizer.from_pretrained("./lora_outputs/lora-model")

    print("Modelo carregado")

In [ ]:
def gerar_texto(prompt, max_tokens=100, temperature=0.7, top_p=0.9):
    
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(inference_model.device)

    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=inference_tokenizer.eos_token_id,
        repetition_penalty=1.2
    )

    texto_gerado = inference_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return texto_gerado

In [ ]:
# Teste 1
prompt1 = "This movie was"
print(f"\nPrompt: '{prompt1}'")
print(f"Resposta: {gerar_texto(prompt1, max_tokens=30)}\n")
print("-------------------------------------------------------------")

# Teste 2
prompt2 = "I really enjoyed"
print(f"\nPrompt: '{prompt2}'")
print(f"Resposta: {gerar_texto(prompt2, max_tokens=30)}\n")
print("-------------------------------------------------------------")

# Teste 3
prompt3 = "The acting was"
print(f"\nPrompt: '{prompt3}'")
print(f"Resposta: {gerar_texto(prompt3, max_tokens=30)}\n")


In [ ]:
import torch

def gerar_com_lambda_simples(
    prompt,
    lambda_value,
    model,
    tokenizer,
    max_new_tokens=50,
    temperature=0.7,
    seed=42
):
    """
    Gera texto aplicando lambda scaling nos logits
    """
    # Seed
    torch.manual_seed(seed)

    # Tokenizar prompt
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    # Loop pra gerar token a token
    for step in range(max_new_tokens):

        # 1. logits base com lora desativado
        with torch.no_grad():
            with model.disable_adapter():
                outputs_base = model(input_ids)
                logits_base = outputs_base.logits[:, -1, :]

        # 2. logits lora
        with torch.no_grad():
            outputs_lora = model(input_ids)
            logits_lora = outputs_lora.logits[:, -1, :]

        # 3. aplica lambda
        delta = logits_lora - logits_base
        logits_final = logits_base + lambda_value * delta

        logits_final = logits_final / temperature
        probs = torch.softmax(logits_final, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # Shape: [1, 1]

        # 5. Adicionar token
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        # 6. Para se for EOS
        if next_token.item() == tokenizer.eos_token_id:
            break

    # Decodificar
    texto_final = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return texto_final

# ========================================
# testes com alguns lambdas
# ========================================

prompt_teste = "This movie was"

print("TESTANDO DIFERENTES LAMBDAS")
print(f"Prompt: '{prompt_teste}'")

lambdas_teste = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

for lambda_val in lambdas_teste:
    print(f"lambda = {lambda_val}")

    resultado = gerar_com_lambda_simples(
        prompt=prompt_teste,
        lambda_value=lambda_val,
        model=inference_model,
        tokenizer=inference_tokenizer,
        max_new_tokens=40,
        temperature=0.7,
        seed=72
    )

    print(resultado)


In [ ]:
# EXPERIMENTO COMPLETO: 7 lambdas × 20 Seeds × 3 Prompts
# TOTAL = 420 gerações

import torch
import pandas as pd
from datetime import datetime
import time

lambdas = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
seeds = [42, 72, 123, 456, 789, 111, 222, 333, 444, 555, 11, 99, 2024, 314, 777, 1985, 2003, 2008, 2019, 2026]
prompts = [
    "This movie was",
    "The acting was",
    "I really enjoyed"
]

def detectar_artefatos(texto):
    """
    Conta artefatos HTML e outros problemas
    """
    artefatos_html = ['<br />', '<br/>', '</br>', '</', '/>']
    count = sum(texto.count(art) for art in artefatos_html)

    return {
        'tem_html': count > 0,
        'count_html': count,
        'tem_advertisement': 'Advertisement' in texto or 'Continue reading' in texto
    }

def calcular_metricas_basicas(texto):
    """
    Métricas básicas do texto gerado
    """
    tokens = inference_tokenizer.encode(texto)
    palavras = texto.split()

    return {
        'num_tokens': len(tokens),
        'num_palavras': len(palavras),
        'palavras_unicas': len(set(palavras)),
        'diversidade_lexical': len(set(palavras)) / len(palavras) if palavras else 0
    }

# INICIO DO EXPERIMENTO


print("INICIANDO EXPERIMENTO COMPLETO")
print(f"Configuração:")
print(f"   - Lambdas: {lambdas}")
print(f"   - Seeds: {seeds}")
print(f"   - Prompts: {len(prompts)}")
print(f"   - Total de gerações: {len(lambdas) * len(seeds) * len(prompts)}")
print("========================================")
print()

resultados = []
total_combinacoes = len(lambdas) * len(seeds) * len(prompts)
contador = 0
inicio_experimento = time.time()

# Loop principal
for prompt in prompts:
    for seed_val in seeds:
        for lambda_val in lambdas:
            contador += 1

            # Barra de progresso
            progresso = (contador / total_combinacoes) * 100
            print(f"[{contador:3d}/{total_combinacoes}] ({progresso:5.1f}%) | "
                  f"Prompt: '{prompt[:15]:15}...' | λ={lambda_val:.1f} | Seed={seed_val}")

            # Gerar texto
            inicio = time.time()

            texto_gerado = gerar_com_lambda_simples(
                prompt=prompt,
                lambda_value=lambda_val,
                model=inference_model,
                tokenizer=inference_tokenizer,
                max_new_tokens=50,
                temperature=0.7,
                seed=seed_val
            )

            tempo_geracao = time.time() - inicio

            # Calcula métricas
            artefatos = detectar_artefatos(texto_gerado)
            metricas = calcular_metricas_basicas(texto_gerado)

            # Armazenar resultado
            resultado = {
                'prompt': prompt,
                'seed': seed_val,
                'lambda': lambda_val,
                'texto_gerado': texto_gerado,
                'tempo_geracao_seg': round(tempo_geracao, 2),
                'num_tokens': metricas['num_tokens'],
                'num_palavras': metricas['num_palavras'],
                'diversidade_lexical': round(metricas['diversidade_lexical'], 3),
                'tem_artefatos_html': artefatos['tem_html'],
                'count_artefatos': artefatos['count_html'],
                'tem_advertisement': artefatos['tem_advertisement'],
            }

            resultados.append(resultado)

            # Preview
            preview = texto_gerado[:60] + "..." if len(texto_gerado) > 60 else texto_gerado
            print(f"         {preview}")
            print()

# ========================================
# SALVAR RESULTADOS
# ========================================

tempo_total = time.time() - inicio_experimento

print("========================================")
print("EXPERIMENTO CONCLUÍDO!")
print("========================================")
print(f"Tempo total: {tempo_total/60:.1f} minutos")
print(f"Total de gerações: {len(resultados)}")
print()

# Converte para DataFrame
df = pd.DataFrame(resultados)

# Salva CSV
filename = f"experimento_lambda_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(filename, index=False)
print(f"Resultados salvos em: {filename}")
print()

# ========================================
# RELATÓRIO RESUMIDO
# ========================================

print("========================================")
print("RELATÓRIO RESUMIDO")
print("========================================")
print()

# Artefatos por lambda
print("1 - ARTEFATOS HTML POR λ:")

artefatos_por_lambda = df.groupby('lambda')['tem_artefatos_html'].agg(['sum', 'count'])
artefatos_por_lambda['percentual'] = (artefatos_por_lambda['sum'] / artefatos_por_lambda['count'] * 100).round(1)

for lambda_val in lambdas:
    row = artefatos_por_lambda.loc[lambda_val]
    total = int(row['count'])
    com_artefato = int(row['sum'])
    pct = row['percentual']

    barra = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))

    print(f"λ={lambda_val:.1f}: {com_artefato:2d}/{total:2d} ({pct:5.1f}%) {barra}")

print()

# Tamanho médio por lambda
print("2 - TAMANHO MÉDIO (tokens) POR λ:")
tamanho_por_lambda = df.groupby('lambda')['num_tokens'].agg(['mean', 'std'])

for lambda_val in lambdas:
    row = tamanho_por_lambda.loc[lambda_val]
    media = row['mean']
    desvio = row['std']

    print(f"λ={lambda_val:.1f}: {media:5.1f} ± {desvio:4.1f} tokens")

print()

# Diversidade léxical por lambda
print("3 - DIVERSIDADE LEXICAL POR λ:")
diversidade_por_lambda = df.groupby('lambda')['diversidade_lexical'].agg(['mean', 'std'])

for lambda_val in lambdas:
    row = diversidade_por_lambda.loc[lambda_val]
    media = row['mean']
    desvio = row['std']

    barra = "█" * int(media * 20) + "░" * (20 - int(media * 20))

    print(f"λ={lambda_val:.1f}: {media:.3f} ± {desvio:.3f} {barra}")

print()

# Consistência entre seeds
print("4 - VARIAÇÃO ENTRE SEEDS (por λ):")
print("Medindo: Desvio padrão do tamanho de texto entre seeds")
print()

for lambda_val in lambdas:
    subset = df[df['lambda'] == lambda_val]
    variacao = subset.groupby('seed')['num_tokens'].mean().std()

    nivel = "Baixa" if variacao < 5 else "Média" if variacao < 10 else "Alta"

    print(f"λ={lambda_val:.1f}: Variação = {variacao:5.1f} ({nivel})")

print()
print(f"Dados completos salvos em: {filename}")